# 005 · Lambdas and Higher-Order Functions

> **Syntax › Functions** · Python 3.14

A *higher-order function* takes functions as arguments or returns them. C# developers know the pattern from LINQ and `Func<>`; Python has the same toolbox, but with a much more limited `lambda` and a strong preference for comprehensions over `map`/`filter`.

**In this notebook you will learn:**

- `lambda` expressions and how they differ from C# lambdas
- using `key=` functions with `sorted`, `min` and `max`
- `map`/`filter` vs LINQ `Select`/`Where`, and why comprehensions usually win
- `functools.reduce` vs LINQ `Aggregate`
- `functools.partial` for pre-filling arguments
- the `operator` module (`itemgetter`, `attrgetter`, ...)
- writing functions that return functions

## C# mental model

| C# | Python |
|---|---|
| `x => x * 2` | `lambda x: x * 2` |
| `(a, b) => a + b` | `lambda a, b: a + b` |
| statement lambda `x => { ...; return y; }` | not possible: use a `def` |
| `Func<int, int>`, `Action<string>` | any callable; hint with `Callable[[int], int]` |
| `OrderBy(p => p.Price)` | `sorted(items, key=lambda p: p["price"])` |
| `Select` / `Where` | `map` / `filter`, or better, a comprehension |
| `Aggregate` | `functools.reduce` |
| `p => p.Name` | `operator.attrgetter("name")` |

## Lambda: a single expression

`lambda params: expression` creates an anonymous function whose body is **one expression**, whose value is returned automatically. There are no statements: no assignments (except `:=`), no `return`, no multi-line blocks, no `try`.

```csharp
// C#
Func<int, int> square = x => x * x;
Func<int, int, int> add = (a, b) => a + b;
Func<string> hello = () => "hello";
```

In [ ]:
square = lambda x: x * x
add = lambda a, b: a + b
hello = lambda: "hello"

print(square(5), add(2, 3), hello())
print(square.__name__)       # '<lambda>': anonymous, which hurts tracebacks

Lambdas support defaults, `*args`, and conditional expressions, but anything with real logic should become a named `def`. Style guides (PEP 8) also discourage assigning a lambda to a name, as done above for demonstration: use `def` then, since you get a proper `__name__`.

In [ ]:
greet = lambda name, punctuation="!": f"Hi {name}{punctuation}"
classify = lambda n: "even" if n % 2 == 0 else "odd"

print(greet("alice"), greet("bob", "?"))
print([classify(n) for n in range(4)])

## Type hints: `Callable`

The equivalent of `Func<int, int>` is `Callable[[int], int]` (argument types in a list, then the return type). `Action<string>` becomes `Callable[[str], None]`. Import it from `collections.abc` *(see Type Hints › 001 · Type Annotations)*.

In [ ]:
from collections.abc import Callable


def apply_twice(func: Callable[[int], int], value: int) -> int:
    return func(func(value))


print(apply_twice(lambda x: x + 3, 10))
print(apply_twice(square, 3))

## `key=` functions: `sorted`, `min`, `max`

The most common place you will write a lambda. The `key` function is called once per item, and its result is used for comparison, exactly like a LINQ key selector.

```csharp
// C#
var cheapest  = products.MinBy(p => p.Price);
var byName    = products.OrderBy(p => p.Name).ToList();
var byStockDesc = products.OrderByDescending(p => p.Stock).ThenBy(p => p.Name).ToList();
```

In [ ]:
products = [
    {"name": "pen", "price": 1.5, "stock": 40},
    {"name": "notebook", "price": 4.0, "stock": 15},
    {"name": "mug", "price": 8.5, "stock": 15},
]

print(min(products, key=lambda p: p["price"])["name"])     # MinBy
print(max(products, key=lambda p: p["price"])["name"])     # MaxBy
print([p["name"] for p in sorted(products, key=lambda p: p["name"])])

# ThenBy: return a tuple. Negate numbers for descending order.
print([p["name"] for p in sorted(products, key=lambda p: (-p["stock"], p["name"]))])

## `map` and `filter` vs `Select` and `Where`

`map(func, iterable)` applies a function to each item; `filter(func, iterable)` keeps items where the function returns truthy. Both return **lazy iterators**, like LINQ's deferred execution, so wrap them in `list()` to materialise *(see Iterators & Generators › 003 · Generator Expressions and Lazy Evaluation)*.

```csharp
// C#
var doubled = numbers.Select(n => n * 2).ToList();
var evens   = numbers.Where(n => n % 2 == 0).ToList();
```

In [ ]:
numbers = [1, 2, 3, 4, 5, 6]

doubled = map(lambda n: n * 2, numbers)
print(doubled)                  # a lazy map object, not a list
print(list(doubled))

evens = list(filter(lambda n: n % 2 == 0, numbers))
print(evens)

print(list(map(str.upper, ["a", "b"])))   # map shines with an existing function

In idiomatic Python, a **comprehension** is usually preferred over `map`/`filter` with a lambda: it reads left to right, combines select and filter in one expression, and avoids the lambda call overhead *(see Collections › 007 · Comprehensions)*.

In [ ]:
# Where(...).Select(...) in one readable expression
print([n * n for n in numbers if n % 2 == 0])

# The map/filter version of the same thing
print(list(map(lambda n: n * n, filter(lambda n: n % 2 == 0, numbers))))

⚠️ **Gotcha:** `map` and `filter` return **one-shot iterators**. Once consumed they are empty, unlike a LINQ query, which re-runs every time you enumerate it.

In [ ]:
evens_iter = filter(lambda n: n % 2 == 0, numbers)
print(list(evens_iter))   # [2, 4, 6]
print(list(evens_iter))   # []: already exhausted

Many LINQ aggregates are plain built-ins in Python: `sum`, `min`, `max`, `len`, `any` (LINQ `Any`), `all` (LINQ `All`), `sorted`, `reversed`, `zip`, `enumerate`.

In [ ]:
print(sum(numbers), any(n > 5 for n in numbers), all(n > 0 for n in numbers))

## `functools.reduce` vs `Aggregate`

`reduce(func, iterable, initial)` folds a sequence into one value, like LINQ's `Aggregate`. It was demoted from a built-in to `functools` because an explicit loop (or `sum`, `max`, `math.prod`, `str.join`) is usually clearer.

```csharp
// C#
var product = numbers.Aggregate(1, (acc, n) => acc * n);
```

In [ ]:
from functools import reduce
import math

product = reduce(lambda acc, n: acc * n, numbers, 1)
print(product)
print(math.prod(numbers))        # the clearer built-in alternative

longest = reduce(lambda a, b: a if len(a) >= len(b) else b, ["kiwi", "banana", "fig"])
print(longest)                   # better: max(words, key=len)

## `functools.partial`

`partial(func, *args, **kwargs)` returns a new callable with some arguments already filled in. In C# you would write a wrapping lambda (`x => Power(x, 2)`), which also works in Python, but `partial` is explicit, introspectable and avoids the late-binding problem *(see Functions › 004 · Scope and Closures)*.

In [ ]:
from functools import partial


def power(base, exponent):
    return base ** exponent


square_it = partial(power, exponent=2)
cube_it = partial(power, exponent=3)
print(square_it(5), cube_it(2))

parse_binary = partial(int, base=2)      # works with any callable, including built-ins
print(parse_binary("1011"))
print(square_it.func.__name__, square_it.keywords)

## The `operator` module

`operator` provides fast, named functions for common operations, so you don't have to write trivial lambdas:

| Lambda | `operator` equivalent |
|---|---|
| `lambda d: d["price"]` | `itemgetter("price")` |
| `lambda d: (d["a"], d["b"])` | `itemgetter("a", "b")` |
| `lambda o: o.name` | `attrgetter("name")` |
| `lambda a, b: a + b` | `operator.add` |
| `lambda o: o.strip()` | `methodcaller("strip")` |

In [ ]:
from operator import itemgetter, attrgetter, methodcaller
import operator

print([p["name"] for p in sorted(products, key=itemgetter("price"))])
print(sorted(products, key=itemgetter("stock", "name"))[0]["name"])   # multi-key sort
print(reduce(operator.add, [1, 2, 3, 4]))
print(list(map(methodcaller("strip"), ["  a ", " b"])))

`attrgetter` does the same for attributes, including dotted paths, which makes it ideal for sorting objects. (`@dataclass` just generates a simple class here; see *Classes & OOP › 008 · Dataclasses*.)

In [ ]:
from dataclasses import dataclass


@dataclass
class User:
    name: str
    age: int


users = [User("carol", 35), User("alice", 30), User("bob", 30)]
print([u.name for u in sorted(users, key=attrgetter("age", "name"))])
print(max(users, key=attrgetter("age")))

## Returning functions

A higher-order function can also **build** functions. This is a closure factory *(see Functions › 004 · Scope and Closures)*, the Python counterpart of a C# method returning `Func<...>`.

```csharp
// C#
static Func<decimal, decimal> MakeDiscount(decimal percent) => price => price * (1 - percent / 100);
```

In [ ]:
def make_discount(percent: float) -> Callable[[float], float]:
    def apply(price: float) -> float:
        return round(price * (1 - percent / 100), 2)
    return apply


ten_off = make_discount(10)
half_price = make_discount(50)
print(ten_off(80.0), half_price(80.0))

And functions that both take and return functions let you compose behaviour. Decorators build directly on this idea *(see Functions › 006 · Decorators)*:

In [ ]:
def compose(*funcs):
    """compose(f, g, h)(x) == f(g(h(x)))"""
    def composed(x):
        for f in reversed(funcs):
            x = f(x)
        return x
    return composed


slugify = compose(lambda s: s.replace(" ", "-"), str.lower, str.strip)
print(slugify("  Hello World  "))

## Summary

| Task | C# | Python |
|---|---|---|
| Anonymous function | `x => x * 2` | `lambda x: x * 2` |
| Multi-statement lambda | `x => { ... }` | use `def` |
| Delegate type | `Func<int, int>` | `Callable[[int], int]` |
| Sort by key | `OrderBy(p => p.Price)` | `sorted(xs, key=itemgetter("price"))` |
| Min/max by key | `MinBy` / `MaxBy` | `min(xs, key=...)` / `max(xs, key=...)` |
| Project / filter | `Select` / `Where` | comprehension (or `map` / `filter`) |
| Fold | `Aggregate` | `functools.reduce` (or a built-in like `sum`) |
| Pre-fill arguments | `x => F(x, 2)` | `partial(f, y=2)` |
| Property selector | `p => p.Name` | `attrgetter("name")` |

## Exercises

1. Given `words = ["banana", "Kiwi", "apple", "fig"]`, find the shortest word with `min` and sort the list by last letter.
2. Rewrite `list(map(lambda w: w.upper(), filter(lambda w: len(w) > 3, words)))` as a comprehension.
3. Use `partial` to create `log_error` from `def log(message, level="INFO"): return f"[{level}] {message}"`.
4. Write `make_adder(n)` that returns a function adding `n` to its argument, then use it with `map` to add 100 to `[1, 2, 3]`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
words = ["banana", "Kiwi", "apple", "fig"]

# 1
print(min(words, key=len))
print(sorted(words, key=lambda w: w[-1]))

# 2
print([w.upper() for w in words if len(w) > 3])

# 3
from functools import partial

def log(message, level="INFO"):
    return f"[{level}] {message}"

log_error = partial(log, level="ERROR")
print(log_error("disk full"))

# 4
def make_adder(n):
    return lambda x: x + n

print(list(map(make_adder(100), [1, 2, 3])))
```

</details>